In [39]:
# importing the libraries needed
import pandas as pd
import plotly.express as px
import numpy as np
import requests as rq
import bs4
from io import StringIO

In [54]:
# reading the webpage data
url = 'https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467'
page = rq.get(url, headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15"})
page.text[0:99]
# parsing the webpage
soup = bs4.BeautifulSoup(page.text, 'html.parser')

In [64]:
# reading the webpage data and extracting tables
tables = pd.read_html(StringIO(str(soup)))
# for loop to print columns
for i in range(len(tables)):
    df =tables[i]
    print(df.columns)

Index([0], dtype='int64')
Index([0, 1, 2], dtype='int64')
MultiIndex([( 'Country/Territory', 'Country/Territory'),
            (         'UN region',         'UN region'),
            (        'IMF[1][13]',          'Forecast'),
            (        'IMF[1][13]',              'Year'),
            (    'World Bank[14]',          'Estimate'),
            (    'World Bank[14]',              'Year'),
            ('United Nations[15]',          'Estimate'),
            ('United Nations[15]',              'Year')],
           )
Index(['vteLists of countries by financial rankings', 'vteLists of countries by financial rankings.1'], dtype='str')
Index(['vteLists of countries by GDP rankings', 'vteLists of countries by GDP rankings.1'], dtype='str')
Index(['vteEconomic classification of countries', 'vteEconomic classification of countries.1'], dtype='str')
Index([0, 1], dtype='int64')


In [65]:
# selecting the third table 
df = tables[2]
df.head()

Country/Territory UN region IMF[1][13]            World Bank[14]             \
  Country/Territory UN region   Forecast       Year       Estimate       Year   
0             World         —  104476432       2023      100562011       2022   
1     United States  Americas   26949643       2023       25462700       2022   
2             China      Asia   17700899  [n 1]2023       17963171  [n 3]2022   
3           Germany    Europe    4429838       2023        4072192       2022   
4             Japan      Asia    4230862       2023        4231141       2022   

  United Nations[15]             
            Estimate       Year  
0           96698005       2021  
1           23315081       2021  
2           17734131  [n 1]2021  
3            4259935       2021  
4            4940878       2021

In [66]:
# selecting the columns we need
clean_df = df.iloc[:, [0, 1, 2, 4, 6]].copy()
clean_df.columns = ['Country', 'Region', 'IMF', 'WorldBank', 'UN']
clean_df.head()

,Country,Region,IMF,WorldBank,UN
0,World,—,104476432,100562011,96698005
1,United States,Americas,26949643,25462700,23315081
2,China,Asia,17700899,17963171,17734131
3,Germany,Europe,4429838,4072192,4259935
4,Japan,Asia,4230862,4231141,4940878


In [ ]:
# types for the data
clean_df.dtypes

Country      str
Region       str
IMF          str
WorldBank    str
UN           str
dtype: object

In [68]:
# converting to numeric values
for col in ['IMF', 'WorldBank', 'UN']:
    clean_df[col] = (
        clean_df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

clean_df.head()

,Country,Region,IMF,WorldBank,UN
0,World,—,104476432.0,100562011.0,96698005.0
1,United States,Americas,26949643.0,25462700.0,23315081.0
2,China,Asia,17700899.0,17963171.0,17734131.0
3,Germany,Europe,4429838.0,4072192.0,4259935.0
4,Japan,Asia,4230862.0,4231141.0,4940878.0


In [74]:
# checking for missing values
print("\nMissing values")
print(clean_df[['IMF', 'WorldBank', 'UN']].isna().sum())


Missing values
IMF          22
WorldBank    27
UN            1
dtype: int64


In [76]:
# remove the world row
clean_df = clean_df[clean_df['Country'] != 'World'].copy()

# fills in the missing data from Worldbank and UN
clean_df['IMF_imputed'] = clean_df['IMF'].fillna(clean_df['WorldBank']).fillna(clean_df['UN'])

# check to make sure no more missing vals
print("Missing values in IMF_imputed:", clean_df['IMF_imputed'].isna().sum())
clean_df.head()

Missing values in IMF_imputed: 0


,Country,Region,IMF,WorldBank,UN,IMF_imputed
1,United States,Americas,26949643.0,25462700.0,23315081.0,26949643.0
2,China,Asia,17700899.0,17963171.0,17734131.0,17700899.0
3,Germany,Europe,4429838.0,4072192.0,4259935.0,4429838.0
4,Japan,Asia,4230862.0,4231141.0,4940878.0,4230862.0
5,India,Asia,3732224.0,3385090.0,3201471.0,3732224.0


In [ ]:

fig = px.bar(data, x='Country/Territory', y='IMF (2026)[1]', color='UN Regions')

fig.show()

ValueError: Value of 'color' is not the name of a column in 'data_frame'. Expected one of ['Country/Territory', 'IMF (2026)[1]', 'World Bank (2025)[6]', 'United Nations (2024)[7]'] but received: UN Regions